In [0]:
%sql
-- Ingestion
SELECT *
FROM csv.`s3://databricks-coursera-thinknyx/` 
WITH (header = true);

In [0]:
%sql
CREATE TABLE workspace.default.walmart_dataset
USING DELTA
LOCATION 's3://databricks-coursera-thinknyx/bronze'
AS
SELECT *,
    current_timestamp() AS created_at,
    input_file_name() AS source_path,
    uuid() AS record_id
FROM csv.`s3://databricks-coursera-thinknyx/`
WITH (header = true, inferSchema=true);

In [0]:
%sql
---- we are adding a new column to find out the 
---- temperature category
alter table workspace.default.walmart_dataset
add column temp_cat string;


In [0]:
%sql
---- now after adding the column set the temperature
UPDATE workspace.default.walmart_dataset
SET temp_cat = 
    CASE
        WHEN Temperature < 40 THEN 'Cold'
        WHEN Temperature BETWEEN 40 AND 70 THEN 'Mild'
        ELSE 'Hot'
    END;

In [0]:
%sql
-- to time travel
-- for verion 3
SELECT * 
FROM workspace.default.walmart_dataset 
VERSION AS OF 3
WHERE Holiday_Flag = 1; 

-- for version 0
SELECT *
FROM workspace.default.walmart_dataset
VERSION AS OF 0
WHERE Holiday_Flag = 1;
